In [4]:
import pandas as pd
from docx import Document
import os
from docx.shared import Pt
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
import re


def fill_dorm_form(template_path, student_pair, output_dir):
    doc = Document(template_path)
    tables = doc.tables
    if len(tables) < 2:
        raise ValueError("Word模板中必须包含至少两个表格（两个凭单）")

    fill_single_form(tables[0], student_pair[0])
    if len(student_pair) > 1:
        fill_single_form(tables[1], student_pair[1])

    os.makedirs(output_dir, exist_ok=True)
    student1 = student_pair[0]
    student2 = student_pair[1] if len(student_pair) > 1 else None
    if student2:
        filename = f"{student1['学工号']}-{student1['姓名']}&{student2['学工号']}-{student2['姓名']}.docx"
    else:
        filename = f"{student1['学工号']}-{student1['姓名']}.docx"

    output_path = os.path.join(output_dir, filename)
    doc.save(output_path)
    return output_path


def fill_single_form(table, student_data):
    for row in table.rows:
        for idx, cell in enumerate(row.cells):
            text = cell.text.strip()
            if "姓  名" in text:
                set_cell_value(row.cells[idx + 1], student_data['姓名'])
            elif "楼  号" in text:
                set_cell_value(row.cells[idx + 1], student_data['楼栋'])
            elif "房间号" in text:
                set_cell_value(row.cells[idx + 1], student_data['房间'])
            elif "床位号" in text:
                set_cell_value(row.cells[idx + 1], student_data['床位'])


def set_cell_value(cell, value):
    cell.text = str(value)
    for paragraph in cell.paragraphs:
        paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
        for run in paragraph.runs:
            run.font.size = Pt(10.5)


def clean_bed_number(bed_str):
    match = re.search(r'\d+', bed_str)
    return match.group() if match else bed_str


def clean_room_number(room_str):
    match = re.search(r'\d+', room_str)
    return match.group() if match else room_str


def clean_building_number(building_str):
    match = re.search(r'(\d+[^\d\s]?)', building_str)
    return match.group(1) if match else (building_str[-5:] if len(building_str) > 5 else building_str)


def process_dorm_forms(excel_path, template_path, output_dir):
    df = pd.read_excel(excel_path)

    # 必填字段
    required_fields = ['学工号', '姓名', '楼栋', '房间', '床位']
    for field in required_fields:
        if field not in df.columns:
            raise ValueError(f"Excel缺少必需列: {field}")

    success_count, error_count, skip_count = 0, 0, 0
    student_groups = [df.iloc[i:i + 2] for i in range(0, len(df), 2)]

    for group_index, group in enumerate(student_groups):
        try:
            student_pair = []
            skip_group = False

            for student_index, (_, row) in enumerate(group.iterrows()):
                if any(pd.isna(row[field]) or str(row[field]).strip() == "" for field in required_fields):
                    print(f"⚠ 跳过组 {group_index+1} 学生 {student_index+1}：存在缺失数据")
                    skip_group = True
                    break

                student_data = {
                    '学工号': str(row['学工号']).strip(),
                    '姓名': str(row['姓名']).strip(),
                    '楼栋': clean_building_number(str(row['楼栋']).strip()),
                    '房间': clean_room_number(str(row['房间'])),
                    '床位': clean_bed_number(str(row['床位']))
                }

                print(f"组 {group_index+1} - 学生 {student_index+1}: {student_data}")
                student_pair.append(student_data)

            if skip_group:
                skip_count += 1
                continue

            output_path = fill_dorm_form(template_path, student_pair, output_dir)
            print(f"✅ 已生成: {output_path}\n")
            success_count += 1

        except Exception as e:
            print(f"❌ 处理组 {group_index+1} 时出错: {str(e)}")
            import traceback
            traceback.print_exc()
            error_count += 1

    print("\n处理完成!")
    print(f"成功生成: {success_count} 个文件")
    print(f"失败: {error_count} 个")
    print(f"跳过: {skip_count} 个 (缺失数据)")


if __name__ == "__main__":
    EXCEL_PATH = 'data.xlsx'
    TEMPLATE_PATH = 'doc.docx'
    OUTPUT_DIR = '住宿凭单'
    process_dorm_forms(EXCEL_PATH, TEMPLATE_PATH, OUTPUT_DIR)
    # 执行处理
    process_dorm_forms(EXCEL_PATH, TEMPLATE_PATH, OUTPUT_DIR)

组 1 - 学生 1: {'学工号': '2500000001', '姓名': '张三', '楼栋': '1楼', '房间': '102', '床位': '3'}
组 1 - 学生 2: {'学工号': '2500000002', '姓名': '李四', '楼栋': '1楼', '房间': '101', '床位': '3'}
✅ 已生成: 住宿凭单/2500000001-张三&2500000002-李四.docx


处理完成!
成功生成: 1 个文件
失败: 0 个
跳过: 0 个 (缺失数据)
组 1 - 学生 1: {'学工号': '2500000001', '姓名': '张三', '楼栋': '1楼', '房间': '102', '床位': '3'}
组 1 - 学生 2: {'学工号': '2500000002', '姓名': '李四', '楼栋': '1楼', '房间': '101', '床位': '3'}
✅ 已生成: 住宿凭单/2500000001-张三&2500000002-李四.docx


处理完成!
成功生成: 1 个文件
失败: 0 个
跳过: 0 个 (缺失数据)
